In [ ]:
!pip install -q demucs fastapi uvicorn python-multipart soundfile

import os, re, uuid, glob, shutil, threading, subprocess, time, asyncio
from fastapi import FastAPI, Request, HTTPException, BackgroundTasks
from fastapi.responses import FileResponse
import uvicorn

app = FastAPI()

@app.get("/health")
def health():
    return {"status": "ok", "gpu": True}

def run_demucs_and_mp3(in_path: str, out_dir: str, mp3_out: str):
    subprocess.run(
        ["demucs", "--two-stems=vocals", "-d", "cuda", "-o", out_dir, in_path],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.PIPE,
        check=True
    )
    matches = glob.glob(os.path.join(out_dir, "**", "no_vocals.wav"), recursive=True)
    if not matches:
        raise RuntimeError("no_vocals.wav not generated")
    # Convert 42MB WAV -> 3.7MB 128k MP3 for fast download
    subprocess.run(
        ["ffmpeg", "-y", "-i", matches[0], "-b:a", "128k", mp3_out],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=True
    )

@app.post("/separate")
async def separate(request: Request, background_tasks: BackgroundTasks):
    form = await request.form()
    upload = next((val for val in form.values() if hasattr(val, "filename")), None)
    if upload is None:
        raise HTTPException(status_code=400, detail="No audio file uploaded")

    job_id = str(uuid.uuid4())[:8]
    work_dir = f"/tmp/demucs_{job_id}"
    os.makedirs(work_dir, exist_ok=True)
    in_path = os.path.join(work_dir, "input_audio")
    out_dir = os.path.join(work_dir, "out")
    mp3_out = os.path.join(work_dir, "no_vocals.mp3")

    with open(in_path, "wb") as f:
        f.write(await upload.read())

    print(f"[{job_id}] Processing chunk on GPU...", flush=True)
    try:
        await asyncio.to_thread(run_demucs_and_mp3, in_path, out_dir, mp3_out)
    except Exception as e:
        shutil.rmtree(work_dir, ignore_errors=True)
        print(f"[{job_id}] ERROR: {e}", flush=True)
        raise HTTPException(status_code=500, detail=f"Demucs failed: {e}")

    print(f"[{job_id}] Done -> returning MP3", flush=True)
    background_tasks.add_task(shutil.rmtree, work_dir, True)
    return FileResponse(mp3_out, media_type="audio/mpeg", filename="no_vocals.mp3")

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

threading.Thread(target=run_server, daemon=True).start()
time.sleep(2)

# Run Pinggy with -T (no TUI spam) and print ONLY the tunnel URLs once
proc = subprocess.Popen(
    [
        "ssh", "-T",
        "-o", "StrictHostKeyChecking=no",
        "-o", "ServerAliveInterval=30",
        "-p", "443",
        "-R0:127.0.0.1:8000",
        "a.pinggy.io"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

printed_urls = set()
print("Starting Pinggy Tunnel...", flush=True)
for line in proc.stdout:
    urls = re.findall(r"https://[a-zA-Z0-9.-]+\.(?:run\.pinggy-free\.link|free\.pinggy\.net)", line)
    for u in urls:
        if u not in printed_urls:
            printed_urls.add(u)
            print(f"✅ Tunnel Ready: {u}", flush=True)
    if len(printed_urls) >= 2:
        # Drain remaining output silently so buffer never blocks and browser never crashes
        threading.Thread(target=lambda p: [None for _ in p.stdout], args=(proc,), daemon=True).start()
        break

print("Server is active (60 min session). Keep this cell running.", flush=True)
proc.wait()

Starting Pinggy Tunnel...
✅ Tunnel Ready: https://qpvot-136-119-27-189.run.pinggy-free.link
✅ Tunnel Ready: https://wbwsy-136-119-27-189.free.pinggy.net
Server is active (60 min session). Keep this cell running.
[6494b3f5] Processing chunk on GPU...
[6494b3f5] Done -> returning MP3
[32edad8a] Processing chunk on GPU...
[32edad8a] Done -> returning MP3
[27da1fa9] Processing chunk on GPU...
[27da1fa9] Done -> returning MP3
[4229dae2] Processing chunk on GPU...
[4229dae2] Done -> returning MP3
[c0b8623c] Processing chunk on GPU...
[c0b8623c] Done -> returning MP3
[5df4edec] Processing chunk on GPU...
[5df4edec] Done -> returning MP3
[75d98cf0] Processing chunk on GPU...
[75d98cf0] Done -> returning MP3
[25774944] Processing chunk on GPU...
[25774944] Done -> returning MP3
[44f674cc] Processing chunk on GPU...
